All the imports need to run the training code. 
Make sure you are using the specified versions or you will run into conflicts. 

In [1]:
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, Tuple
from tqdm.auto import tqdm

import math
import os
import time
from collections import OrderedDict, defaultdict
from random import random, uniform, randint

import numpy as np
import segmentation_models_pytorch as smp
import torch
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader, Dataset, Sampler, random_split
from torch.utils.tensorboard import SummaryWriter
from torchvision.io import read_image
from torch.utils.data import Subset

### CONFIG CELL
Edit this block instead of changing values all over the file.

In [2]:
@dataclass
class ModelConfig:
    architecture: str = "unet"            # "unet" or "deeplabv3plus"
    encoder_name: str = "resnet34"
    encoder_weights: Optional[str] = "imagenet"
    in_channels: int = 3
    num_classes: int = 5
    freeze_batchnorm: bool = True


@dataclass
class DataConfig:
    root_dir: str = "/home/nels/TeamVida/Tick Dataset"
    patch_size: int = 512
    overlap: int = 100
    train_val_split: float = 0.2
    split_seed: int = 42
    train_cache_capacity: int = 2
    val_cache_capacity: int = 2
    rotate_mask_if_swapped: bool = True
    train_ignore_edge_band: int = 1
    val_ignore_edge_band: int = 0
    num_workers_train: int = 6
    num_workers_val: int = 4
    pin_memory: bool = True
    prefetch_factor_train: int = 4
    prefetch_factor_val: int = 2
    train_batch_size: int = 8
    val_batch_size: int = 8
    class_index_min_pixels: int = 512
    class_sampler_per_class: int = 1

@dataclass
class CVConfig:
    enabled: bool = True
    num_folds: int = 5
    seed: int = 42
    shuffle: bool = True


@dataclass
class AugConfig:
    use_strong_aug: bool = True
    ignore_index: int = 255


@dataclass
class LossConfig:
    ignore_index: int = 255
    ce_weight: float = 0.4
    ft_weight: float = 0.6
    edge_aux_weight: float = 0.03
    focal_tversky_classes: Tuple[int, ...] = (1, 2, 3)
    focal_tversky_alpha: Tuple[float, ...] = (0.70, 0.60, 0.60)
    focal_tversky_beta: Tuple[float, ...] = (0.42, 0.40, 0.40)
    focal_tversky_gamma: float = 1.25
    effective_num_beta: float = 0.999
    bg_scale: float = 0.40
    litter_scale: float = 0.22



@dataclass
class TrainConfig:
    epochs: int = 25
    patience: int = 5
    lr: float = 2e-4
    weight_decay: float = 1e-5
    amp_enabled: bool = True
    amp_dtype: torch.dtype = torch.float16
    use_ema: bool = True
    ema_decay: float = 0.996
    ema_warmup_fraction_of_epoch: float = 1.0
    scheduler_factor: float = 0.1
    scheduler_patience: int = 3

@dataclass
class LoggingConfig:
    run_name_suffix: str = "unet_r34_ps512"
    log_dir: str = "runs"
    save_best_live_path: str = "best_by_miou_exbg_live.pth"
    save_best_ema_path: str = "best_by_miou_exbg_ema.pth"


@dataclass
class AppConfig:
    class_names: Tuple[str, ...] = ("BG", "Forb", "Graminoid", "Woody", "Litter")
    color_map: dict = field(default_factory=lambda: {
        (0, 0, 0): 0,
        (74, 222, 128): 1,
        (96, 165, 250): 2,
        (29, 78, 216): 2,
        (221, 255, 51): 3,
        (220, 38, 38): 4,
    })
    index_to_color: dict = field(default_factory=lambda: {
        0: (0, 0, 0),
        1: (74, 222, 128),
        2: (96, 165, 250),
        3: (221, 255, 51),
        4: (220, 38, 38),
    })
    model: ModelConfig = field(default_factory=ModelConfig)
    data: DataConfig = field(default_factory=DataConfig)
    aug: AugConfig = field(default_factory=AugConfig)
    loss: LossConfig = field(default_factory=LossConfig)
    train: TrainConfig = field(default_factory=TrainConfig)
    logging: LoggingConfig = field(default_factory=LoggingConfig)
    cv: CVConfig = field(default_factory=CVConfig)


CFG = AppConfig()

### Helpers

In [3]:
def log_per_class(prefix: str, ious, names=CFG.class_names):
    line = "  ".join(f"{n}={iou * 100:.2f}%" for n, iou in zip(names, ious))
    print(f"{prefix} per-class IoU:  {line}")


def build_dense_lut(color_map: dict):
    lut = np.zeros((256, 256, 256), dtype=np.uint8)
    for (r, g, b), cls in color_map.items():
        lut[r, g, b] = np.uint8(cls)
    return lut


def rgb_to_classes_dense(mask_rgb_chw: torch.Tensor, dense_lut: np.ndarray) -> torch.Tensor:
    m = mask_rgb_chw.permute(1, 2, 0).contiguous().cpu().numpy()
    cls = dense_lut[m[..., 0], m[..., 1], m[..., 2]]
    return torch.from_numpy(cls)

class _LRU:
    def __init__(self, capacity: int):
        self.cap = capacity
        self._d = OrderedDict()

    def get(self, k):
        v = self._d.get(k)
        if v is not None:
            self._d.move_to_end(k)
        return v

    def set(self, k, v):
        self._d[k] = v
        self._d.move_to_end(k)
        if len(self._d) > self.cap:
            self._d.popitem(last=False)


@dataclass
class SampleMeta:
    img_path: str
    mask_path: str
    W: int
    H: int

import numpy as np

def make_kfold_splits(dataset_len, num_folds=5, seed=42, shuffle=True):
    indices = np.arange(dataset_len)
    if shuffle:
        rng = np.random.default_rng(seed)
        rng.shuffle(indices)

    folds = np.array_split(indices, num_folds)

    split_list = []
    for fold_idx in range(num_folds):
        val_idx = folds[fold_idx].tolist()
        train_idx = np.concatenate([folds[i] for i in range(num_folds) if i != fold_idx]).tolist()
        split_list.append((train_idx, val_idx))
    return split_list


### Dataset utilities 


In [4]:
class VegetationDataset(Dataset):
    def __init__(self, root_dir: str):
        self.samples = []
        root = Path(root_dir)
        for folder in root.iterdir():
            if not folder.is_dir():
                continue
            img = folder / f"{folder.name}.jpg"
            msk = folder / f"{folder.name}_mask.png"
            if img.exists() and msk.exists():
                self.samples.append((str(img), str(msk)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]
    
@torch.no_grad()
def add_ignore_band(msk_u8: torch.Tensor, band: int = 1, ignore_index: int = 255) -> torch.Tensor:
    t = msk_u8.to(torch.long)
    base = t.clone()
    base[base == ignore_index] = 0

    edge = torch.zeros_like(base, dtype=torch.bool)
    edge[1:, :] |= (base[1:, :] != base[:-1, :])
    edge[:-1, :] |= (base[:-1, :] != base[1:, :])
    edge[:, 1:] |= (base[:, 1:] != base[:, :-1])
    edge[:, :-1] |= (base[:, :-1] != base[:, 1:])

    if band > 0:
        e = F.max_pool2d(edge.float().unsqueeze(0).unsqueeze(0), kernel_size=2 * band + 1, stride=1, padding=band)
        e = e.squeeze(0).squeeze(0).bool()
    else:
        e = edge

    out = t.clone()
    out[e] = ignore_index
    return out.to(torch.uint8)

def _gaussian_blur(img_f, sigma):
    k = max(3, int(2 * math.ceil(3 * sigma) + 1))
    x = torch.arange(k, device=img_f.device) - k // 2
    g1d = torch.exp(-0.5 * (x.float() / sigma) ** 2)
    g1d = g1d / g1d.sum()

    kw = g1d.view(1, 1, 1, k).expand(3, 1, 1, k)
    kh = g1d.view(1, 1, k, 1).expand(3, 1, k, 1)

    img_f = F.conv2d(img_f, kw, padding=(0, k // 2), groups=3)
    img_f = F.conv2d(img_f, kh, padding=(k // 2, 0), groups=3)
    return img_f

def _resize_pair(img_u8, msk_u8, scale, out_h, out_w, ignore_index=255):
    H, W = msk_u8.shape
    new_h, new_w = max(1, int(H * scale)), max(1, int(W * scale))

    img_f = F.interpolate(img_u8.unsqueeze(0).float(), size=(new_h, new_w), mode='bilinear', align_corners=False)
    msk_i = F.interpolate(msk_u8.unsqueeze(0).unsqueeze(0).float(), size=(new_h, new_w), mode='nearest')
    msk_i = msk_i.squeeze(0).squeeze(0).to(torch.uint8)

    pad_h = max(0, out_h - new_h)
    pad_w = max(0, out_w - new_w)
    if pad_h or pad_w:
        pad = (pad_w // 2, pad_w - pad_w // 2, pad_h // 2, pad_h - pad_h // 2)
        img_f = F.pad(img_f, pad, mode='reflect')
        msk_i = F.pad(msk_i.unsqueeze(0).unsqueeze(0), pad, mode='constant', value=ignore_index).squeeze(0).squeeze(0)

    h, w = img_f.shape[2], img_f.shape[3]
    y0 = 0 if h == out_h else randint(0, h - out_h)
    x0 = 0 if w == out_w else randint(0, w - out_w)
    img_f = img_f[:, :, y0:y0 + out_h, x0:x0 + out_w]
    msk_i = msk_i[y0:y0 + out_h, x0:x0 + out_w]
    img_u8 = img_f.squeeze(0).clamp(0, 255).to(torch.uint8)
    return img_u8, msk_i

def _cutout_inplace(img_u8, num_holes=1, max_frac=0.20):
    _, H, W = img_u8.shape
    for _ in range(num_holes):
        f = uniform(0.05, max_frac)
        h = max(1, int(H * f))
        w = max(1, int(W * f))
        y0 = randint(0, max(0, H - h))
        x0 = randint(0, max(0, W - w))
        img_u8[:, y0:y0 + h, x0:x0 + w] = 0
    return img_u8

def aug_stronger(img_u8, msk_u8, ignore_index=255):
    H, W = msk_u8.shape

    if random() < 0.5:
        img_u8 = torch.flip(img_u8, [2])
        msk_u8 = torch.flip(msk_u8, [1])
    if random() < 0.5:
        img_u8 = torch.flip(img_u8, [1])
        msk_u8 = torch.flip(msk_u8, [0])
    k = randint(0, 3)
    if k:
        img_u8 = torch.rot90(img_u8, k, [1, 2])
        msk_u8 = torch.rot90(msk_u8, k, [0, 1])

    if random() < 0.9:
        s = uniform(0.75, 1.30)
        img_u8, msk_u8 = _resize_pair(img_u8, msk_u8, s, H, W, ignore_index=ignore_index)

    if random() < 0.9:
        gain = (0.8 + 0.4 * torch.rand(3, 1, 1)).to(img_u8.device)
        img_u8 = (img_u8.float() * gain).clamp(0, 255).to(torch.uint8)

    if random() < 0.7:
        img_f = img_u8.float()
        b = uniform(-0.20, 0.20) * 255.0
        c = uniform(0.8, 1.2)
        mean = img_f.mean(dim=(1, 2), keepdim=True)
        img_f = (img_f - mean) * c + mean + b
        img_u8 = img_f.clamp(0, 255).to(torch.uint8)

    if random() < 0.3:
        delta = torch.randn(3, 1, 1, device=img_u8.device) * 0.03
        img_u8 = (img_u8.float() * (1.0 + delta)).clamp(0, 255).to(torch.uint8)

    if random() < 0.2:
        sigma = uniform(0.6, 1.2)
        img_f = (img_u8.float() / 255.0).unsqueeze(0)
        img_f = _gaussian_blur(img_f, sigma)
        img_u8 = (img_f.squeeze(0).clamp(0, 1) * 255.0).to(torch.uint8)

    if random() < 0.15:
        img_u8 = _cutout_inplace(img_u8, num_holes=randint(1, 2), max_frac=0.15)

    return img_u8.contiguous(), msk_u8.contiguous()

### Dataset Class

In [5]:
class PatchDataset(Dataset):
    def __init__(self, image_ds_or_subset, color_map: dict, patch_size: int, overlap: int,
                 cache_capacity: int, rotate_mask_if_swapped: bool, ignore_edge_band: int, augment=None):
        self.patch_size = patch_size
        self.step = patch_size - overlap
        self.rotate_mask_if_swapped = rotate_mask_if_swapped
        self.ignore_edge_band = int(ignore_edge_band)
        self.augment = augment

        if hasattr(image_ds_or_subset, 'dataset') and hasattr(image_ds_or_subset, 'indices'):
            base_ds = image_ds_or_subset.dataset
            indices = list(image_ds_or_subset.indices)
        else:
            base_ds = image_ds_or_subset
            indices = list(range(len(base_ds)))

        if not hasattr(base_ds, 'samples'):
            raise TypeError("Base dataset must expose `.samples = [(img_path, mask_path), ...]`")

        self.samples = []
        for i in indices:
            img_path, mask_path = base_ds.samples[i]
            with Image.open(img_path) as im:
                W, H = im.size
            self.samples.append(SampleMeta(img_path, mask_path, W, H))

        self.image_ids = list(range(len(self.samples)))
        self.dense_lut = build_dense_lut(color_map)
        self.index = []
        self.grouped_patch_indices = defaultdict(list)

        ps = self.patch_size
        for img_id, s in enumerate(self.samples):
            for y in range(0, s.H - ps + 1, self.step):
                for x in range(0, s.W - ps + 1, self.step):
                    self.grouped_patch_indices[img_id].append(len(self.index))
                    self.index.append((img_id, y, x))

        if not self.index:
            raise RuntimeError("No patches generated. Check patch_size/overlap against image sizes.")

        self._img_cache = _LRU(cache_capacity)
        self._mask_cache = _LRU(cache_capacity)

    def __len__(self):
        return len(self.index)

    def _load_image_uint8_chw(self, path: str) -> torch.Tensor:
        t = self._img_cache.get(path)
        if t is None:
            t = read_image(path)
            if t.ndim != 3 or t.shape[0] != 3:
                raise RuntimeError(f"Expected 3-channel image at {path}, got {tuple(t.shape)}")
            self._img_cache.set(path, t)
        return t

    def _load_mask_classes(self, meta: SampleMeta) -> torch.Tensor:
        t = self._mask_cache.get(meta.mask_path)
        if t is None:
            mask_rgb = read_image(meta.mask_path)
            if mask_rgb.ndim != 3 or mask_rgb.shape[0] != 3:
                raise RuntimeError(f"Expected 3-channel mask at {meta.mask_path}, got {tuple(mask_rgb.shape)}")
            Hm, Wm = mask_rgb.shape[1], mask_rgb.shape[2]
            if self.rotate_mask_if_swapped and (Hm, Wm) != (meta.H, meta.W) and (Wm, Hm) == (meta.H, meta.W):
                mask_rgb = torch.rot90(mask_rgb, k=1, dims=(1, 2))
                Hm, Wm = mask_rgb.shape[1], mask_rgb.shape[2]
            if (Hm, Wm) != (meta.H, meta.W):
                raise RuntimeError(f"Mask size {Hm}x{Wm} != image size {meta.H}x{meta.W}\n  {meta.mask_path}")
            t = rgb_to_classes_dense(mask_rgb, self.dense_lut).to(torch.uint8)
            self._mask_cache.set(meta.mask_path, t)
        return t

    def __getitem__(self, idx):
        image_id, y, x = self.index[idx]
        meta = self.samples[image_id]
        ps = self.patch_size

        img = self._load_image_uint8_chw(meta.img_path)
        msk = self._load_mask_classes(meta)

        img_patch = img[:, y:y + ps, x:x + ps].contiguous()
        msk_patch = msk[y:y + ps, x:x + ps].contiguous()

        if self.augment is not None:
            img_patch, msk_patch = self.augment(img_patch, msk_patch)
            img_patch = img_patch.to(torch.uint8)
            msk_patch = msk_patch.to(torch.uint8)

        if self.ignore_edge_band > 0:
            msk_patch = add_ignore_band(msk_patch, band=self.ignore_edge_band, ignore_index=CFG.loss.ignore_index)

        return img_patch, msk_patch, image_id

### Sampler + Split


In [6]:
def create_train_val_datasets(full_dataset, val_ratio=0.2, seed=42):
    dataset_len = len(full_dataset)
    val_len = int(dataset_len * val_ratio)
    train_len = dataset_len - val_len
    generator = torch.Generator().manual_seed(seed)
    return random_split(full_dataset, [train_len, val_len], generator=generator)


class ImageGroupedBatchSampler(Sampler):
    def __init__(self, patch_dataset, batch_size: int, shuffle_images=True, shuffle_patches=True, drop_last=False, seed=None):
        self.ds = patch_dataset
        self.batch_size = int(batch_size)
        self.shuffle_images = shuffle_images
        self.shuffle_patches = shuffle_patches
        self.drop_last = drop_last
        self.seed = seed
        self.image_ids = list(self.ds.grouped_patch_indices.keys())

    def __iter__(self):
        rng = np.random.default_rng(self.seed)
        image_ids = self.image_ids.copy()
        if self.shuffle_images:
            rng.shuffle(image_ids)

        for img_id in image_ids:
            idxs = self.ds.grouped_patch_indices[img_id].copy()
            if self.shuffle_patches:
                rng.shuffle(idxs)
            bs = self.batch_size
            n = len(idxs)
            full = (n // bs) * bs
            for i in range(0, full, bs):
                yield idxs[i:i + bs]
            rem = n - full
            if rem and not self.drop_last:
                yield idxs[full:]

    def __len__(self):
        total = 0
        for img_id in self.image_ids:
            n = len(self.ds.grouped_patch_indices[img_id])
            total += n // self.batch_size if self.drop_last else math.ceil(n / self.batch_size)
        return total


class ClassAwareBatchSampler(Sampler):
    def __init__(self, by_class, fg_pool, batch_size=8, per_class=1, seed=42, fill_with_fg=True, drop_last=False):
        self.by = {c: np.array(v, dtype=np.int64) for c, v in by_class.items()}
        self.classes = list(self.by.keys())
        pool = fg_pool if (fg_pool and fill_with_fg) else sum(by_class.values(), [])
        self.pool = np.array(pool, dtype=np.int64)
        self.bs = int(batch_size)
        self.kc = int(per_class)
        self.seed = int(seed)
        self.drop_last = drop_last
        self._epoch = 0

    def set_epoch(self, epoch: int):
        self._epoch = int(epoch)

    def __len__(self):
        base = len(self.pool) if len(self.pool) > 0 else sum(len(v) for v in self.by.values())
        return max(1, base // self.bs if self.drop_last else (base + self.bs - 1) // self.bs)

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self._epoch)
        for _ in range(len(self)):
            batch = []
            for c in self.classes:
                arr = self.by[c]
                if len(arr) == 0:
                    continue
                take = self.kc
                replace = len(arr) < take
                sel = rng.choice(arr, size=take if replace else min(take, len(arr)), replace=replace)
                batch.extend(sel.tolist())

            rem = self.bs - len(batch)
            if rem > 0:
                pool = self.pool if len(self.pool) else np.concatenate(list(self.by.values())) if len(self.by) else np.array([], dtype=np.int64)
                if len(pool) > 0:
                    replace = len(pool) < rem
                    fill = rng.choice(pool, size=rem if replace else min(rem, len(pool)), replace=replace)
                    batch.extend(fill.tolist())

            rng.shuffle(batch)
            if self.drop_last and len(batch) < self.bs:
                continue
            yield batch[:self.bs]


def simple_collate(batch):
    imgs = torch.stack([b[0] for b in batch], dim=0)
    masks = torch.stack([b[1] for b in batch], dim=0)
    img_ids = torch.tensor([b[2] for b in batch], dtype=torch.long)
    return imgs, masks, img_ids


@torch.no_grad()
def build_class_index(patch_ds, classes=(1, 2, 3), min_pixels=256, ignore_index=None, disable_aug=True):
    prev_aug = None
    if disable_aug and hasattr(patch_ds, 'augment'):
        prev_aug = patch_ds.augment
        patch_ds.augment = None

    try:
        by_class = {c: [] for c in classes}
        fg_pool = []
        for i in range(len(patch_ds)):
            _, m, _ = patch_ds[i]
            v = m.reshape(-1)
            if ignore_index is not None:
                v = v[v != ignore_index]
            has_any = False
            for c in classes:
                if (v == c).sum().item() >= min_pixels:
                    by_class[c].append(i)
                    has_any = True
            if has_any:
                fg_pool.append(i)
        return by_class, fg_pool
    finally:
        if prev_aug is not None:
            patch_ds.augment = prev_aug

### Models + Loss

In [7]:
def build_model(cfg: AppConfig):
    if cfg.model.architecture.lower() == 'unet':
        model = smp.Unet(
            encoder_name=cfg.model.encoder_name,
            encoder_weights=cfg.model.encoder_weights,
            in_channels=cfg.model.in_channels,
            classes=cfg.model.num_classes,
        )
    elif cfg.model.architecture.lower() == 'deeplabv3plus':
        model = smp.DeepLabV3Plus(
            encoder_name=cfg.model.encoder_name,
            encoder_weights=cfg.model.encoder_weights,
            in_channels=cfg.model.in_channels,
            classes=cfg.model.num_classes,
            encoder_output_stride=8,
        )
    else:
        raise ValueError(f"Unsupported architecture: {cfg.model.architecture}")

    if cfg.model.freeze_batchnorm:
        for m in model.modules():
            if isinstance(m, torch.nn.BatchNorm2d):
                m.eval()
                for p in m.parameters():
                    p.requires_grad = False
    return model


@torch.no_grad()
def compute_class_freqs_with_loader(patch_ds, num_classes, batch_size=64, num_workers=4):
    loader = DataLoader(
        patch_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=False,
        collate_fn=lambda batch: [m for (_, m, _) in batch],
    )

    counts = torch.zeros(num_classes, dtype=torch.long)
    for masks in loader:
        for m in masks:
            v = m.reshape(-1).to(torch.long)
            if CFG.loss.ignore_index is not None:
                v = v[v != CFG.loss.ignore_index]
            counts += torch.bincount(v, minlength=num_classes)[:num_classes]
    counts_np = counts.cpu().numpy()
    freqs_np = counts_np / max(counts_np.sum(), 1)
    return counts_np, freqs_np


def effective_number_weights(counts, num_classes=5, plant_classes=(1, 2, 3), beta=0.999, bg_scale=0.40, litter_scale=0.22, device='cpu'):
    counts = torch.as_tensor(counts, dtype=torch.float32, device=device).clamp_min(1.0)
    weights = (1.0 - beta) / (1.0 - torch.pow(beta, counts))
    out = torch.ones(num_classes, device=device)
    plant_idx = torch.tensor(plant_classes, device=device)
    plant_w = weights[plant_idx]
    out[plant_idx] = plant_w / plant_w.mean().clamp_min(1e-12)
    out[0] = bg_scale
    if num_classes > 4:
        out[4] = litter_scale
    return out


_sobel_cache = {}


@torch.no_grad()
def sobel_kernels(device, dtype):
    key = (device, dtype)
    if key not in _sobel_cache:
        kx = torch.tensor([[1, 0, -1], [2, 0, -2], [1, 0, -1]], dtype=dtype, device=device).view(1, 1, 3, 3)
        ky = torch.tensor([[1, 2, 1], [0, 0, 0], [-1, -2, -1]], dtype=dtype, device=device).view(1, 1, 3, 3)
        _sobel_cache[key] = (kx, ky)
    return _sobel_cache[key]


def focal_tversky_loss(logits, targets, classes=(1, 2, 3), alpha=(0.6, 0.6, 0.6), beta=(0.4, 0.4, 0.4), gamma=1.25, ignore_index=255, eps=1e-7):
    N, C, H, W = logits.shape
    probs = logits.softmax(1)
    valid = (targets != ignore_index) if ignore_index is not None else torch.ones_like(targets, dtype=torch.bool)
    tgt = targets.clone()
    tgt[~valid] = 0
    onehot = torch.nn.functional.one_hot(tgt.clamp(min=0), num_classes=C).permute(0, 3, 1, 2).float()
    onehot = onehot * valid.unsqueeze(1)

    idx = torch.tensor(classes, device=logits.device)
    p = probs[:, idx]
    g = onehot[:, idx]

    dims = (0, 2, 3)
    tp = (p * g).sum(dims)
    fp = (p * (1 - g)).sum(dims)
    fn = ((1 - p) * g).sum(dims)

    a = torch.tensor(alpha, device=logits.device).view(-1)
    b = torch.tensor(beta, device=logits.device).view(-1)
    tversky = (tp + eps) / (tp + a * fn + b * fp + eps)
    return (1.0 - tversky).pow(gamma).mean()


def edge_aux_loss(logits, targets, ignore_index=255, eps=1e-6):
    probs = logits.softmax(1)
    plant_p = 1.0 - (probs[:, 0:1] + probs[:, 4:5])
    valid = (targets != ignore_index).unsqueeze(1)
    gt_plant = ((targets != 0) & (targets != 4) & (targets != ignore_index)).unsqueeze(1).to(plant_p.dtype)
    kx, ky = sobel_kernels(logits.device, logits.dtype)
    gx_p = F.conv2d(plant_p, kx, padding=1)
    gy_p = F.conv2d(plant_p, ky, padding=1)
    grad_p = torch.sqrt(gx_p * gx_p + gy_p * gy_p + eps)
    gx_g = F.conv2d(gt_plant, kx, padding=1)
    gy_g = F.conv2d(gt_plant, ky, padding=1)
    grad_g = torch.sqrt(gx_g * gx_g + gy_g * gy_g + eps)
    valid = valid.to(plant_p.dtype)
    return (valid * (grad_p - grad_g).abs()).sum() / valid.sum().clamp_min(1)


class CombinedCriterion(torch.nn.Module):
    def __init__(self, ce_weights: torch.Tensor, cfg: AppConfig):
        super().__init__()
        self.ce_weights = ce_weights
        self.cfg = cfg

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(
            logits,
            targets,
            weight=self.ce_weights,
            ignore_index=self.cfg.loss.ignore_index,
            reduction='mean',
        )
        ft_loss = focal_tversky_loss(
            logits,
            targets,
            classes=self.cfg.loss.focal_tversky_classes,
            alpha=self.cfg.loss.focal_tversky_alpha,
            beta=self.cfg.loss.focal_tversky_beta,
            gamma=self.cfg.loss.focal_tversky_gamma,
            ignore_index=self.cfg.loss.ignore_index,
        )
        e_loss = edge_aux_loss(logits, targets, ignore_index=self.cfg.loss.ignore_index)
        return self.cfg.loss.ce_weight * ce_loss + self.cfg.loss.ft_weight * ft_loss + self.cfg.loss.edge_aux_weight * e_loss

### Evaluation Metrics

In [8]:
def intersection_and_union(pred_or_logits, target, num_classes, ignore_index=None, eps=1e-6):
    with torch.no_grad():
        pred = pred_or_logits.argmax(dim=1) if pred_or_logits.dim() == 4 else pred_or_logits
        pred_flat = pred.reshape(-1)
        target_flat = target.reshape(-1)

        if ignore_index is not None:
            valid = target_flat != ignore_index
            pred_flat = pred_flat[valid]
            target_flat = target_flat[valid]

        device = pred.device
        intersection = torch.zeros(num_classes, device=device, dtype=torch.float32)
        union = torch.zeros(num_classes, device=device, dtype=torch.float32)

        for cls in range(num_classes):
            p = (pred_flat == cls)
            t = (target_flat == cls)
            intersection[cls] = (p & t).sum().float()
            union[cls] = (p | t).sum().float()

        iou = (intersection + eps) / (union + eps)
        return intersection, union, iou

### EMA

In [9]:
class EMA:
    def __init__(self, model, base_decay=0.999, warmup_steps=1000):
        self.base_decay = base_decay
        self.warmup_steps = warmup_steps
        self.num_updates = 0
        self.shadow = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
        self.backup = None

    @torch.no_grad()
    def _decay(self):
        if self.warmup_steps <= 0:
            return self.base_decay
        t = min(1.0, self.num_updates / float(self.warmup_steps))
        return self.base_decay * t

    @torch.no_grad()
    def update(self, model):
        self.num_updates += 1
        d = self._decay()
        for n, p in model.named_parameters():
            if n in self.shadow:
                self.shadow[n].mul_(d).add_(p.detach(), alpha=1.0 - d)

    @torch.no_grad()
    def store(self, model):
        self.backup = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}

    @torch.no_grad()
    def copy_to(self, model):
        for n, p in model.named_parameters():
            if n in self.shadow:
                p.data.copy_(self.shadow[n].data)

    @torch.no_grad()
    def restore(self, model):
        if self.backup is None:
            return
        for n, p in model.named_parameters():
            if n in self.backup:
                p.data.copy_(self.backup[n].data)
        self.backup = None

### Train / Validate

In [10]:
def _get_norm_buffers(device):
    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
    return mean, std


def train_one_epoch(model, dataloader, optimizer, criterion, device, num_classes,
                    scaler=None, amp_dtype=torch.float16, amp_enabled=True,
                    ema=None, ignore_index=None, epoch=None, total_epochs=None):
    model.train()
    mean, std = _get_norm_buffers(device)

    running_loss = 0.0
    running_correct = 0
    total_pixels = 0
    total_intersection = torch.zeros(num_classes, device=device)
    total_union = torch.zeros(num_classes, device=device)

    pbar = tqdm(
        dataloader,
        desc=f"Train {epoch+1}/{total_epochs}" if epoch is not None and total_epochs is not None else "Training",
        leave=False
    )

    for images, masks, _ in pbar:
        images = images.to(device=device, dtype=torch.float32, non_blocking=True,
                           memory_format=torch.channels_last)
        images.mul_(1 / 255.0).sub_(mean).div_(std)
        masks = masks.to(device, non_blocking=True).long()

        optimizer.zero_grad(set_to_none=True)

        with autocast(dtype=amp_dtype, enabled=amp_enabled):
            outputs = model(images)
            loss = criterion(outputs, masks)

        if scaler is not None and amp_enabled:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        if ema is not None:
            ema.update(model)

        B = images.size(0)
        running_loss += loss.item() * B
        preds = outputs.argmax(dim=1)

        valid = torch.ones_like(masks, dtype=torch.bool) if ignore_index is None else (masks != ignore_index)
        running_correct += ((preds == masks) & valid).sum().item()
        total_pixels += valid.sum().item()

        intersection, union, _ = intersection_and_union(preds, masks, num_classes, ignore_index=ignore_index)
        total_intersection += intersection
        total_union += union

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc": f"{100.0 * running_correct / max(total_pixels, 1):.2f}%"
        })

    avg_loss = running_loss / len(dataloader.dataset)
    avg_acc = 100.0 * running_correct / max(total_pixels, 1)
    iou_per_class = (total_intersection / (total_union + 1e-6)).detach().cpu().numpy()
    mean_iou = float(iou_per_class.mean())
    return avg_loss, avg_acc, mean_iou, iou_per_class


@torch.no_grad()
def validate_one_epoch(model, dataloader, criterion, device, num_classes,
                       amp_dtype=torch.float16, amp_enabled=True,
                       ignore_index=None, epoch=None, total_epochs=None):
    model.eval()
    mean, std = _get_norm_buffers(device)

    running_loss = 0.0
    running_correct = 0
    total_pixels = 0
    total_intersection = torch.zeros(num_classes, device=device)
    total_union = torch.zeros(num_classes, device=device)

    pbar = tqdm(
        dataloader,
        desc=f"Val {epoch+1}/{total_epochs}" if epoch is not None and total_epochs is not None else "Validating",
        leave=False
    )

    for images, masks, _ in pbar:
        images = images.to(device=device, dtype=torch.float32, non_blocking=True,
                           memory_format=torch.channels_last)
        images.mul_(1 / 255.0).sub_(mean).div_(std)
        masks = masks.to(device, non_blocking=True).long()

        with autocast(dtype=amp_dtype, enabled=amp_enabled):
            outputs = model(images)
            loss = criterion(outputs, masks)

        B = images.size(0)
        running_loss += loss.item() * B
        preds = outputs.argmax(dim=1)

        valid = torch.ones_like(masks, dtype=torch.bool) if ignore_index is None else (masks != ignore_index)
        running_correct += ((preds == masks) & valid).sum().item()
        total_pixels += valid.sum().item()

        intersection, union, _ = intersection_and_union(preds, masks, num_classes, ignore_index=ignore_index)
        total_intersection += intersection
        total_union += union

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc": f"{100.0 * running_correct / max(total_pixels, 1):.2f}%"
        })

    avg_loss = running_loss / len(dataloader.dataset)
    avg_acc = 100.0 * running_correct / max(total_pixels, 1)
    iou_per_class = (total_intersection / (total_union + 1e-6)).detach().cpu().numpy()
    mean_iou = float(iou_per_class.mean())
    return avg_loss, avg_acc, mean_iou, iou_per_class


@torch.no_grad()
def validate_tta(model, dataloader, criterion, device, num_classes, amp_dtype=torch.float16, amp_enabled=True, ignore_index=None):
    model.eval()
    mean, std = _get_norm_buffers(device)

    running_loss = 0.0
    running_correct = 0
    total_pixels = 0
    total_intersection = torch.zeros(num_classes, device=device)
    total_union = torch.zeros(num_classes, device=device)

    for images, masks, _ in dataloader:
        images = images.to(device=device, dtype=torch.float32, non_blocking=True, memory_format=torch.channels_last)
        images.mul_(1 / 255.0).sub_(mean).div_(std)
        masks = masks.to(device, non_blocking=True).long()

        with autocast(enabled=amp_enabled, dtype=amp_dtype):
            o0 = model(images)
            o1 = torch.flip(model(torch.flip(images, dims=[-1])), dims=[-1])
            o2 = torch.flip(model(torch.flip(images, dims=[-2])), dims=[-2])
            o3 = torch.flip(model(torch.flip(images, dims=[-2, -1])), dims=[-2, -1])
            logits = (o0 + o1 + o2 + o3) / 4.0
            loss = criterion(logits, masks)

        B = images.size(0)
        running_loss += loss.item() * B
        preds = logits.argmax(dim=1)

        valid = torch.ones_like(masks, dtype=torch.bool) if ignore_index is None else (masks != ignore_index)
        running_correct += ((preds == masks) & valid).sum().item()
        total_pixels += valid.sum().item()

        inter, uni, _ = intersection_and_union(preds, masks, num_classes, ignore_index=ignore_index)
        total_intersection += inter
        total_union += uni

    avg_loss = running_loss / len(dataloader.dataset)
    avg_acc = 100.0 * running_correct / max(total_pixels, 1)
    iou_per_class = (total_intersection / (total_union + 1e-6)).detach().cpu().numpy()
    mean_iou = float(iou_per_class.mean())
    return avg_loss, avg_acc, mean_iou, iou_per_class

### Setup Cell

In [11]:
def build_everything(cfg: AppConfig, train_indices=None, val_indices=None):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = True

    full_dataset = VegetationDataset(cfg.data.root_dir)

    if train_indices is None or val_indices is None:
        train_dataset, val_dataset = create_train_val_datasets(
            full_dataset,
            val_ratio=cfg.data.train_val_split,
            seed=cfg.data.split_seed,
        )
    else:
        train_dataset = Subset(full_dataset, train_indices)
        val_dataset = Subset(full_dataset, val_indices)

    train_patches = PatchDataset(
        train_dataset,
        color_map=cfg.color_map,
        patch_size=cfg.data.patch_size,
        overlap=cfg.data.overlap,
        cache_capacity=cfg.data.train_cache_capacity,
        rotate_mask_if_swapped=cfg.data.rotate_mask_if_swapped,
        ignore_edge_band=cfg.data.train_ignore_edge_band,
        augment=aug_stronger if cfg.aug.use_strong_aug else None,
    )

    val_patches = PatchDataset(
        val_dataset,
        color_map=cfg.color_map,
        patch_size=cfg.data.patch_size,
        overlap=cfg.data.overlap,
        cache_capacity=cfg.data.val_cache_capacity,
        rotate_mask_if_swapped=cfg.data.rotate_mask_if_swapped,
        ignore_edge_band=cfg.data.val_ignore_edge_band,
        augment=None,
    )

    pixel_counts, class_freqs = compute_class_freqs_with_loader(
        train_patches,
        num_classes=cfg.model.num_classes,
        batch_size=64,
        num_workers=min(4, cfg.data.num_workers_train),
    )

    ce_weights = effective_number_weights(
        pixel_counts,
        num_classes=cfg.model.num_classes,
        plant_classes=cfg.loss.focal_tversky_classes,
        beta=cfg.loss.effective_num_beta,
        bg_scale=cfg.loss.bg_scale,
        litter_scale=cfg.loss.litter_scale,
        device=device,
    )

    model = build_model(cfg).to(device)
    criterion = CombinedCriterion(ce_weights=ce_weights, cfg=cfg)
    optimizer = optim.Adam(model.parameters(), lr=cfg.train.lr, weight_decay=cfg.train.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=cfg.train.scheduler_factor,
        patience=cfg.train.scheduler_patience,
        verbose=True,
    )

    by_class, fg_pool = build_class_index(
        train_patches,
        classes=cfg.loss.focal_tversky_classes,
        min_pixels=cfg.data.class_index_min_pixels,
        ignore_index=cfg.loss.ignore_index,
    )

    sampler = ClassAwareBatchSampler(
        by_class=by_class,
        fg_pool=fg_pool,
        batch_size=cfg.data.train_batch_size,
        per_class=cfg.data.class_sampler_per_class,
        seed=cfg.data.split_seed,
        fill_with_fg=True,
        drop_last=False,
    )


    val_batch_sampler = ImageGroupedBatchSampler(
        val_patches,
        batch_size=cfg.data.val_batch_size,
        shuffle_images=False,
        shuffle_patches=False,
    )

    train_loader = DataLoader(
        train_patches,
        batch_sampler=sampler,
        num_workers=cfg.data.num_workers_train,
        pin_memory=cfg.data.pin_memory,
        persistent_workers=cfg.data.num_workers_train > 0,
        prefetch_factor=cfg.data.prefetch_factor_train if cfg.data.num_workers_train > 0 else None,
        collate_fn=simple_collate,
    )

    val_loader = DataLoader(
        val_patches,
        batch_sampler=val_batch_sampler,
        num_workers=cfg.data.num_workers_val,
        pin_memory=cfg.data.pin_memory,
        persistent_workers=cfg.data.num_workers_val > 0,
        prefetch_factor=cfg.data.prefetch_factor_val if cfg.data.num_workers_val > 0 else None,
        collate_fn=simple_collate,
    )

    scaler = GradScaler(enabled=(device.type == 'cuda' and cfg.train.amp_enabled))
    ema = EMA(model, base_decay=cfg.train.ema_decay, warmup_steps=max(1, int(len(train_loader) * cfg.train.ema_warmup_fraction_of_epoch))) if cfg.train.use_ema else None

    return {
        'device': device,
        'model': model,
        'criterion': criterion,
        'optimizer': optimizer,
        'scheduler': scheduler,
        'train_loader': train_loader,
        'val_loader': val_loader,
        'scaler': scaler,
        'ema': ema,
        'pixel_counts': pixel_counts,
        'class_freqs': class_freqs,
        'ce_weights': ce_weights,
    }


### Train Loop Cell

In [12]:
def run_training(cfg: AppConfig, train_indices=None, val_indices=None):
    bundle = build_everything(cfg, train_indices=train_indices, val_indices=val_indices)
    device = bundle['device']
    model = bundle['model']
    criterion = bundle['criterion']
    optimizer = bundle['optimizer']
    scheduler = bundle['scheduler']
    train_loader = bundle['train_loader']
    val_loader = bundle['val_loader']
    scaler = bundle['scaler']
    ema = bundle['ema']
    best_val_iou = None

    run_name = f"{time.strftime('%Y%m%d-%H%M%S')}_{cfg.logging.run_name_suffix}"
    writer = SummaryWriter(log_dir=os.path.join(cfg.logging.log_dir, run_name))

    best_miou_exbg = -1.0
    no_improve = 0

    for epoch in range(cfg.train.epochs):
        if hasattr(train_loader.batch_sampler, 'set_epoch'):
            train_loader.batch_sampler.set_epoch(epoch)

        train_loss, train_acc, train_miou, train_iou = train_one_epoch(
            model, train_loader, optimizer, criterion, device,
            num_classes=cfg.model.num_classes,
            scaler=scaler,
            amp_dtype=cfg.train.amp_dtype,
            amp_enabled=cfg.train.amp_enabled,
            ema=ema,
            ignore_index=cfg.loss.ignore_index,
            epoch=epoch,
            total_epochs=cfg.train.epochs,
        )

        val_loss_no_tta, val_acc, _, _ = validate_one_epoch(
            model, val_loader, criterion, device,
            num_classes=cfg.model.num_classes,
            amp_dtype=cfg.train.amp_dtype,
            amp_enabled=cfg.train.amp_enabled,
            ignore_index=cfg.loss.ignore_index,
            epoch=epoch,
            total_epochs=cfg.train.epochs,
        )

        _, _, _, val_iou_live = validate_tta(
            model, val_loader, criterion, device,
            num_classes=cfg.model.num_classes,
            amp_dtype=cfg.train.amp_dtype,
            amp_enabled=cfg.train.amp_enabled,
            ignore_index=cfg.loss.ignore_index,
        )
        miou_exbg_live = float(val_iou_live[1:4].mean())

        val_iou_ema = None
        miou_exbg_ema = -1.0
        if ema is not None and ema.num_updates >= max(1, ema.warmup_steps // 2):
            ema.store(model)
            ema.copy_to(model)
            _, _, _, val_iou_ema = validate_tta(
                model, val_loader, criterion, device,
                num_classes=cfg.model.num_classes,
                amp_dtype=cfg.train.amp_dtype,
                amp_enabled=cfg.train.amp_enabled,
                ignore_index=cfg.loss.ignore_index,
            )
            ema.restore(model)
            miou_exbg_ema = float(val_iou_ema[1:4].mean())

        use_ema = val_iou_ema is not None and miou_exbg_ema > miou_exbg_live
        chosen_miou = miou_exbg_ema if use_ema else miou_exbg_live
        scheduler.step(1.0 - chosen_miou)

        print(f"Epoch {epoch + 1}/{cfg.train.epochs}")
        print(f"  Train => Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%, mIoU: {train_miou:.4f}")
        log_per_class("  Train", train_iou)
        print(f"  Val   => Loss(no-TTA): {val_loss_no_tta:.4f}, Acc: {val_acc:.2f}%")
        print(
            f"  Val mIoU(no-bg) — live: {miou_exbg_live:.4f}"
            + (f", ema: {miou_exbg_ema:.4f}" if val_iou_ema is not None else ", ema: (skipped)")
            + f"  -> used: {'ema' if use_ema else 'live'}"
        )
        log_per_class("  Val(TTA, live)", val_iou_live)
        if val_iou_ema is not None:
            log_per_class("  Val(TTA, ema)", val_iou_ema)

        writer.add_scalar('train/loss', train_loss, epoch)
        writer.add_scalar('train/mIoU', train_miou, epoch)
        writer.add_scalar('val/loss_no_tta', val_loss_no_tta, epoch)
        writer.add_scalars('val/mIoU_no_bg', {'live': miou_exbg_live, 'ema': miou_exbg_ema}, epoch)

        for i, name in enumerate(cfg.class_names):
            writer.add_scalar(f'val/iou_live/{name}', float(val_iou_live[i]), epoch)
            if val_iou_ema is not None:
                writer.add_scalar(f'val/iou_ema/{name}', float(val_iou_ema[i]), epoch)

        for i, pg in enumerate(optimizer.param_groups):
            writer.add_scalar(f'opt/lr_group{i}', pg['lr'], epoch)
        writer.flush()

        if chosen_miou > best_miou_exbg + 1e-4:
            best_miou_exbg = chosen_miou
            best_val_iou = val_iou_ema.copy() if use_ema and val_iou_ema is not None else val_iou_live.copy()
            if use_ema and ema is not None:
                ema.store(model)
                ema.copy_to(model)
                torch.save(model.state_dict(), cfg.logging.save_best_ema_path)
                ema.restore(model)
            else:
                torch.save(model.state_dict(), cfg.logging.save_best_live_path)
            no_improve = 0
            print('  Validation mIoU improved, model saved.')
        else:
            no_improve += 1
            print(f'  No improvement. Early stop counter: {no_improve}/{cfg.train.patience}')
            if no_improve >= cfg.train.patience:
                print('Early stopping triggered.')
                break

    writer.close()
    return {
    "bundle": bundle,
    "best_miou_exbg": best_miou_exbg,
    "best_val_iou": best_val_iou,
    }



In [ ]:
## Run Single Training loop (Good for debugging)
bundle = run_training(CFG)
print('Training complete.')

### Cross Validation Cell

In [14]:
def run_cross_validation(cfg: AppConfig):
    full_dataset = VegetationDataset(cfg.data.root_dir)
    splits = make_kfold_splits(
        dataset_len=len(full_dataset),
        num_folds=cfg.cv.num_folds,
        seed=cfg.cv.seed,
        shuffle=cfg.cv.shuffle,
    )

    fold_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(splits):
        print(f"\n{'='*70}")
        print(f"Fold {fold_idx + 1}/{cfg.cv.num_folds}")
        print(f"Train images: {len(train_idx)} | Val images: {len(val_idx)}")
        print(f"{'='*70}")

        # optional: fold-specific run/model names
        old_suffix = cfg.logging.run_name_suffix
        old_live = cfg.logging.save_best_live_path
        old_ema = cfg.logging.save_best_ema_path

        cfg.logging.run_name_suffix = f"{old_suffix}_fold{fold_idx+1}"
        cfg.logging.save_best_live_path = f"best_fold{fold_idx+1}_live.pth"
        cfg.logging.save_best_ema_path = f"best_fold{fold_idx+1}_ema.pth"

        result = run_training(cfg, train_indices=train_idx, val_indices=val_idx)

        fold_results.append({
            "fold": fold_idx + 1,
            "best_miou_exbg": result["best_miou_exbg"],
            "best_val_iou": result["best_val_iou"],
        })

        # restore names
        cfg.logging.run_name_suffix = old_suffix
        cfg.logging.save_best_live_path = old_live
        cfg.logging.save_best_ema_path = old_ema

    # aggregate
    miou_scores = np.array([r["best_miou_exbg"] for r in fold_results], dtype=np.float64)
    print("\nCross-validation summary")
    print(f"mIoU(no-bg) per fold: {[round(x, 4) for x in miou_scores]}")
    print(f"Mean: {miou_scores.mean():.4f}")
    print(f"Std : {miou_scores.std(ddof=1):.4f}" if len(miou_scores) > 1 else f"Std : 0.0000")

    return fold_results

In [ ]:
### Run Cross validation
fold_results = run_cross_validation(CFG)